In [1]:
# Block 1 - Imports and paths

import os
import numpy as np
import pandas as pd

import cv2
import pydicom

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models


data_path = "/kaggle/input/competitions/rsna-knee-abnormality-detection"


test_series = pd.read_csv(
    f"{data_path}/test_series.csv"
)

sample_submission = pd.read_csv(
    f"{data_path}/sample_submission.csv"
)


label_columns = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]


print("Test series:", test_series.shape)
print("Submission template:", sample_submission.shape)

print("\nSubmission columns:")
print(sample_submission.columns.tolist())

Test series: (15, 5)
Submission template: (3, 13)

Submission columns:
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [2]:
# Block 2 - Find saved Model V2 checkpoints

model_paths = []


for root, dirs, files in os.walk("/kaggle/input"):

    for file in files:

        if (
            file.startswith("model_v2_fold")
            and file.endswith(".pth")
        ):

            model_paths.append(
                os.path.join(
                    root,
                    file
                )
            )


model_paths = sorted(model_paths)


print(
    "Found V2 model files:",
    len(model_paths)
)

for path in model_paths:
    print(path)

Found V2 model files: 4
/kaggle/input/notebooks/klokiiii8887/04d-rsna-model-v2-multiplane/model_v2_fold0.pth
/kaggle/input/notebooks/klokiiii8887/04d-rsna-model-v2-multiplane/model_v2_fold1.pth
/kaggle/input/notebooks/klokiiii8887/04d-rsna-model-v2-multiplane/model_v2_fold2.pth
/kaggle/input/notebooks/klokiiii8887/04d-rsna-model-v2-multiplane/model_v2_fold3.pth


In [3]:
# Block 3 - Test series types

test_series["Series_Type"] = (
    test_series["Anatomical_Plane"].astype(str)
    + "_F"
    + test_series["Fluid_Sensitive"].astype(int).astype(str)
)

print(
    test_series["Series_Type"]
    .value_counts()
)

Series_Type
Sagittal_F0    4
Coronal_F1     3
Sagittal_F1    3
Axial_F1       3
Axial_F0       1
Coronal_F0     1
Name: count, dtype: int64


In [4]:
# Block 4 - ROBUST hidden-test series selection

def count_test_slices(study_id, series_id):

    path = os.path.join(
        data_path,
        "test_series",
        study_id,
        series_id
    )

    try:
        return sum(
            1
            for entry in os.scandir(path)
            if (
                entry.is_file()
                and entry.name.endswith(".dcm")
            )
        )

    except Exception:
        return 0

In [5]:
def select_test_series(
    study_id,
    plane,
    preferred_type,
    fallback_type=None
):

    study_rows = test_series[
        test_series["StudyInstanceUID"] == study_id
    ].copy()

    if len(study_rows) == 0:
        raise ValueError(
            f"No MRI series exist for study {study_id}"
        )

    # ------------------------------------------------
    # Priority 1 - exact preferred sequence
    # ------------------------------------------------
    candidates = study_rows[
        study_rows["Series_Type"] == preferred_type
    ].copy()

    selection_reason = "preferred"


    # ------------------------------------------------
    # Priority 2 - requested fallback sequence
    # ------------------------------------------------
    if (
        len(candidates) == 0
        and fallback_type is not None
    ):

        candidates = study_rows[
            study_rows["Series_Type"] == fallback_type
        ].copy()

        selection_reason = "sequence_fallback"


    # ------------------------------------------------
    # Priority 3 - ANY series from the requested plane
    # ------------------------------------------------
    if len(candidates) == 0:

        candidates = study_rows[
            study_rows["Anatomical_Plane"] == plane
        ].copy()

        selection_reason = "same_plane_fallback"


    # ------------------------------------------------
    # Priority 4 - hidden-test emergency fallback
    #
    # Do NOT crash if an entire plane is missing.
    # Use another available MRI series instead.
    # ------------------------------------------------
    if len(candidates) == 0:

        candidates = study_rows.copy()

        selection_reason = "any_series_emergency"


    candidates["Slice_Count"] = candidates.apply(
        lambda row: count_test_slices(
            row["StudyInstanceUID"],
            row["SeriesInstanceUID"]
        ),
        axis=1
    )


    candidates = candidates.sort_values(
        by=[
            "Slice_Count",
            "SeriesInstanceUID"
        ],
        ascending=[
            False,
            True
        ]
    )


    selected = candidates.iloc[0]


    return {
        "SeriesInstanceUID":
            selected["SeriesInstanceUID"],

        "Series_Type":
            selected["Series_Type"],

        "Slice_Count":
            int(selected["Slice_Count"]),

        "Selection_Reason":
            selection_reason
    }

In [6]:
# Build robust multi-plane test table

test_rows = []


for study_id in sample_submission["StudyInstanceUID"]:

    axial = select_test_series(
        study_id=study_id,
        plane="Axial",
        preferred_type="Axial_F1"
    )


    coronal = select_test_series(
        study_id=study_id,
        plane="Coronal",
        preferred_type="Coronal_F1",
        fallback_type="Coronal_F0"
    )


    sagittal = select_test_series(
        study_id=study_id,
        plane="Sagittal",
        preferred_type="Sagittal_F0",
        fallback_type="Sagittal_F1"
    )


    test_rows.append({

        "StudyInstanceUID": study_id,

        "Axial_SeriesUID":
            axial["SeriesInstanceUID"],

        "Axial_Type":
            axial["Series_Type"],

        "Axial_Reason":
            axial["Selection_Reason"],


        "Coronal_SeriesUID":
            coronal["SeriesInstanceUID"],

        "Coronal_Type":
            coronal["Series_Type"],

        "Coronal_Reason":
            coronal["Selection_Reason"],


        "Sagittal_SeriesUID":
            sagittal["SeriesInstanceUID"],

        "Sagittal_Type":
            sagittal["Series_Type"],

        "Sagittal_Reason":
            sagittal["Selection_Reason"]
    })


test_model_data = pd.DataFrame(
    test_rows
)


print(
    "Selected test studies:",
    len(test_model_data)
)


print("\nAxial selection reasons:")
print(
    test_model_data["Axial_Reason"]
    .value_counts()
)


print("\nCoronal selection reasons:")
print(
    test_model_data["Coronal_Reason"]
    .value_counts()
)


print("\nSagittal selection reasons:")
print(
    test_model_data["Sagittal_Reason"]
    .value_counts()
)

Selected test studies: 3

Axial selection reasons:
Axial_Reason
preferred    3
Name: count, dtype: int64

Coronal selection reasons:
Coronal_Reason
preferred    3
Name: count, dtype: int64

Sagittal selection reasons:
Sagittal_Reason
preferred    3
Name: count, dtype: int64


In [7]:
# Robust DICOM loader for hidden inference

def load_dicom_series(series_path):

    slices = []


    try:
        entries = list(
            os.scandir(series_path)
        )

    except Exception as e:

        print(
            "WARNING: Could not open series:",
            series_path,
            "|",
            str(e)
        )

        return None


    for position, entry in enumerate(entries):

        if not (
            entry.is_file()
            and entry.name.endswith(".dcm")
        ):
            continue


        try:

            ds = pydicom.dcmread(
                entry.path
            )


            try:
                instance_number = int(
                    getattr(
                        ds,
                        "InstanceNumber",
                        position
                    )
                )

            except Exception:
                instance_number = position


            pixel_array = (
                ds.pixel_array
                .astype(np.float32)
            )


            slices.append(
                (
                    instance_number,
                    pixel_array
                )
            )


        except Exception as e:

            print(
                "WARNING: Skipping bad DICOM:",
                entry.path,
                "|",
                str(e)
            )

            continue


    if len(slices) == 0:

        return None


    slices.sort(
        key=lambda x: x[0]
    )


    try:

        volume = np.stack(
            [
                image
                for _, image in slices
            ]
        )

    except Exception:

        return None


    return volume

In [8]:
def normalize_volume(
    volume,
    lower_percentile=1,
    upper_percentile=99
):

    volume = volume.astype(
        np.float32
    )


    nonzero = volume[
        volume > 0
    ]


    if len(nonzero) == 0:

        return np.zeros_like(
            volume,
            dtype=np.float32
        )


    low = np.percentile(
        nonzero,
        lower_percentile
    )

    high = np.percentile(
        nonzero,
        upper_percentile
    )


    if high <= low:

        return np.zeros_like(
            volume,
            dtype=np.float32
        )


    volume = np.clip(
        volume,
        low,
        high
    )


    volume = (
        volume - low
    ) / (
        high - low
    )


    return volume.astype(
        np.float32
    )

In [9]:
def resize_volume(
    volume,
    target_size=(224, 224)
):

    resized = []


    for image in volume:

        image = cv2.resize(
            image,
            target_size,
            interpolation=cv2.INTER_AREA
        )

        resized.append(
            image
        )


    return np.stack(
        resized
    ).astype(
        np.float32
    )

In [10]:
def sample_slices(
    volume,
    target_slices=16
):

    indices = np.linspace(
        0,
        volume.shape[0] - 1,
        target_slices
    )


    indices = np.round(
        indices
    ).astype(int)


    return volume[
        indices
    ]

In [11]:
def preprocess_series(
    series_path,
    target_size=(224, 224),
    target_slices=16
):

    try:

        volume = load_dicom_series(
            series_path
        )


        # Hidden-test emergency case
        if volume is None:

            print(
                "WARNING: Using blank fallback for:",
                series_path
            )

            return np.zeros(
                (
                    target_slices,
                    target_size[1],
                    target_size[0]
                ),
                dtype=np.float32
            )


        volume = normalize_volume(
            volume
        )


        volume = resize_volume(
            volume,
            target_size
        )


        volume = sample_slices(
            volume,
            target_slices
        )


        return volume.astype(
            np.float32
        )


    except Exception as e:

        print(
            "WARNING: Preprocessing failed:",
            series_path,
            "|",
            str(e)
        )


        return np.zeros(
            (
                target_slices,
                target_size[1],
                target_size[0]
            ),
            dtype=np.float32
        )

In [12]:
# Block 6 - Multi-plane test Dataset

class RSNAKneeMultiPlaneTestDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        data_path
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.data_path = data_path


    def __len__(self):

        return len(
            self.df
        )


    def _load_plane(
        self,
        study_id,
        series_id
    ):

        path = os.path.join(
            self.data_path,
            "test_series",
            study_id,
            series_id
        )


        volume = preprocess_series(
            path
        )


        volume = np.expand_dims(
            volume,
            axis=1
        )


        return torch.from_numpy(
            volume
        ).float()


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]


        study_id = row[
            "StudyInstanceUID"
        ]


        axial = self._load_plane(
            study_id,
            row["Axial_SeriesUID"]
        )


        coronal = self._load_plane(
            study_id,
            row["Coronal_SeriesUID"]
        )


        sagittal = self._load_plane(
            study_id,
            row["Sagittal_SeriesUID"]
        )


        return {

            "axial": axial,

            "coronal": coronal,

            "sagittal": sagittal,

            "study_id": study_id
        }

In [13]:
test_dataset = (
    RSNAKneeMultiPlaneTestDataset(
        dataframe=test_model_data,
        data_path=data_path
    )
)


test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


sample = test_dataset[0]


print(
    "Axial:",
    sample["axial"].shape
)

print(
    "Coronal:",
    sample["coronal"].shape
)

print(
    "Sagittal:",
    sample["sagittal"].shape
)

print(
    "Test studies:",
    len(test_dataset)
)

Axial: torch.Size([16, 1, 224, 224])
Coronal: torch.Size([16, 1, 224, 224])
Sagittal: torch.Size([16, 1, 224, 224])
Test studies: 3


In [14]:
weights=None

In [15]:
# Block 7 - Model V2 inference architecture

class KneeModelV2(nn.Module):

    def __init__(
        self,
        num_labels=12
    ):

        super().__init__()


        self.encoder = models.resnet18(
            weights=None
        )


        feature_dim = (
            self.encoder.fc.in_features
        )


        self.encoder.fc = nn.Identity()


        self.classifier = nn.Linear(
            feature_dim * 3,
            num_labels
        )


        self.register_buffer(
            "mean",
            torch.tensor(
                [0.485, 0.456, 0.406]
            ).view(
                1,
                3,
                1,
                1
            )
        )


        self.register_buffer(
            "std",
            torch.tensor(
                [0.229, 0.224, 0.225]
            ).view(
                1,
                3,
                1,
                1
            )
        )


    def encode_plane(
        self,
        x
    ):

        batch_size = x.shape[0]
        num_slices = x.shape[1]


        x = x.reshape(
            batch_size * num_slices,
            1,
            224,
            224
        )


        x = x.repeat(
            1,
            3,
            1,
            1
        )


        x = (
            x - self.mean
        ) / self.std


        features = self.encoder(
            x
        )


        features = features.reshape(
            batch_size,
            num_slices,
            -1
        )


        return features.mean(
            dim=1
        )


    def forward(
        self,
        axial,
        coronal,
        sagittal
    ):

        axial_feature = (
            self.encode_plane(
                axial
            )
        )

        coronal_feature = (
            self.encode_plane(
                coronal
            )
        )

        sagittal_feature = (
            self.encode_plane(
                sagittal
            )
        )


        combined = torch.cat(
            [
                axial_feature,
                coronal_feature,
                sagittal_feature
            ],
            dim=1
        )


        return self.classifier(
            combined
        )

In [16]:
# Block 8 - Load V2 ensemble

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    device
)


models_v2 = []


for path in model_paths:

    model = KneeModelV2(
        num_labels=12
    )


    state = torch.load(
        path,
        map_location=device
    )


    model.load_state_dict(
        state
    )


    model = model.to(
        device
    )


    model.eval()


    models_v2.append(
        model
    )


print(
    "Loaded V2 models:",
    len(models_v2)
)

Device: cuda
Loaded V2 models: 4


In [17]:
# Block 9 - V2 four-fold ensemble inference

all_predictions = []
all_study_ids = []


with torch.no_grad():

    for batch in test_loader:

        axial = batch["axial"].to(
            device,
            non_blocking=True
        )

        coronal = batch["coronal"].to(
            device,
            non_blocking=True
        )

        sagittal = batch["sagittal"].to(
            device,
            non_blocking=True
        )

        study_ids = batch["study_id"]


        fold_predictions = []


        # Run all four trained fold models
        for model in models_v2:

            logits = model(
                axial,
                coronal,
                sagittal
            )

            probabilities = torch.sigmoid(
                logits
            )

            fold_predictions.append(
                probabilities
            )


        # [4 models, batch, 12 targets]
        fold_predictions = torch.stack(
            fold_predictions,
            dim=0
        )


        # Average all four models
        ensemble_predictions = (
            fold_predictions.mean(
                dim=0
            )
        )


        all_predictions.append(
            ensemble_predictions
            .cpu()
            .numpy()
        )


        all_study_ids.extend(
            list(study_ids)
        )


all_predictions = np.concatenate(
    all_predictions,
    axis=0
)


print(
    "Prediction shape:",
    all_predictions.shape
)

print(
    "Studies predicted:",
    len(all_study_ids)
)

print(
    "Minimum probability:",
    all_predictions.min()
)

print(
    "Maximum probability:",
    all_predictions.max()
)

Prediction shape: (3, 12)
Studies predicted: 3
Minimum probability: 0.12749556
Maximum probability: 0.8064776


In [18]:
# Block 10 - Create V2 submission

submission = pd.DataFrame(
    all_predictions,
    columns=label_columns
)


submission.insert(
    0,
    "StudyInstanceUID",
    all_study_ids
)


print(
    "Raw submission shape:",
    submission.shape
)

display(submission)

Raw submission shape: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.410962,0.127496,0.806478,0.392939,0.428778,0.273668,0.439813,0.571228,0.604410,0.382523,0.311486,0.141768
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.285149,0.188654,0.622833,0.377873,0.398788,0.186328,0.306798,0.569496,0.321128,0.234394,0.294104,0.166565
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.338151,0.186158,0.598928,0.403572,0.405639,0.261833,0.360745,0.594753,0.417503,0.225860,0.291764,0.188151


In [19]:

submission = (
    sample_submission[
        ["StudyInstanceUID"]
    ]
    .merge(
        submission,
        on="StudyInstanceUID",
        how="left"
    )
)


submission = submission[
    sample_submission.columns
]


display(submission)

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.410962,0.127496,0.806478,0.392939,0.428778,0.273668,0.439813,0.571228,0.604410,0.382523,0.311486,0.141768
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.285149,0.188654,0.622833,0.377873,0.398788,0.186328,0.306798,0.569496,0.321128,0.234394,0.294104,0.166565
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.338151,0.186158,0.598928,0.403572,0.405639,0.261833,0.360745,0.594753,0.417503,0.225860,0.291764,0.188151


In [20]:
# Block 12 - Submission validation

print(
    "Shape:",
    submission.shape
)


print(
    "Columns correct:",
    submission.columns.tolist()
    ==
    sample_submission.columns.tolist()
)


print(
    "Study IDs correct:",
    submission["StudyInstanceUID"].tolist()
    ==
    sample_submission["StudyInstanceUID"].tolist()
)


missing_predictions = (
    submission[label_columns]
    .isna()
    .sum()
    .sum()
)


print(
    "Missing predictions:",
    missing_predictions
)


minimum_probability = (
    submission[label_columns]
    .min()
    .min()
)


maximum_probability = (
    submission[label_columns]
    .max()
    .max()
)


print(
    "Minimum probability:",
    minimum_probability
)

print(
    "Maximum probability:",
    maximum_probability
)


print(
    "All probabilities valid:",
    (
        minimum_probability >= 0
        and
        maximum_probability <= 1
    )
)

Shape: (3, 13)
Columns correct: True
Study IDs correct: True
Missing predictions: 0
Minimum probability: 0.12749555706977844
Maximum probability: 0.8064776062965393
All probabilities valid: True


In [21]:
# Block 13 - Save Kaggle submission file

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)


print(
    "Model V2 submission.csv created successfully ✅"
)

Model V2 submission.csv created successfully ✅


In [22]:
print(
    pd.read_csv(
        "/kaggle/working/submission.csv"
    ).head()
)

                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.410962  0.127496   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.285149  0.188654   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.338151  0.186158   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.806478          0.392939   0.428778    0.273668  0.439813   
1         0.622833          0.377873   0.398788    0.186328  0.306798   
2         0.598928          0.403572   0.405639    0.261833  0.360745   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.571228   0.604410  0.382523   0.311486  0.141768  
1  0.569496   0.321128  0.234394   0.294104  0.166565  
2  0.594753   0.417503  0.225860   0.291764  0.188151  


In [23]:
emergency_columns = [
    "Axial_Reason",
    "Coronal_Reason",
    "Sagittal_Reason"
]


for column in emergency_columns:

    print(
        "\n",
        column
    )

    print(
        test_model_data[column]
        .value_counts(
            dropna=False
        )
    )


 Axial_Reason
Axial_Reason
preferred    3
Name: count, dtype: int64

 Coronal_Reason
Coronal_Reason
preferred    3
Name: count, dtype: int64

 Sagittal_Reason
Sagittal_Reason
preferred    3
Name: count, dtype: int64
